In [395]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
import ast


In [396]:
ads_base = pd.read_csv('../data/ads.csv')
users_base = pd.read_csv('../data/users.csv')

users= users_base.copy()
ads = ads_base.copy()

In [397]:
users.head(1)

,user_id,age,gender,state,city,preferences
0,0,58,F,MG,Juiz de Fora,"[('Sports_Games', 0.06144033651140886), ('Console_Gaming', 0.16123710711415512), ('MMORPG', 0.17928985804561), ('VR_Gaming', 0.016847073302291835), ('Minecraft', 0.03901932791588827), ('Fortnite', 0.3253089551775492), ('Esports', 0.04964746680902405), ('Counter_Strike', 0.07772577069622656), ('Game_Hardware', 0.0894841044278461)]"


In [398]:
ads.head(1)

,ad_id,state_target,city_target,age_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate
0,0,PR,Maringá,60,F,"{'Indie_Games': 0.16518794390557134, 'Open_World': 0.060332325664619484, 'PC_Gaming': 0.013830634132233021, 'Simulation': 0.09308320786155601, 'League_of_Legends': 0.13348803361729053, 'VR_Gaming': 0.22099246159470362, 'Retro_Gaming': 0.3130853932240261}",15.0,7500.0,0.940094,0.016979


In [399]:
type(ads.categories)

pandas.core.series.Series

In [400]:
type(users.preferences)

pandas.core.series.Series

In [401]:
# CSV perde a informação de tipo, então precisamos converter as strings de volta para listas e para dicionários. Para isso, podemos usar a função ast.literal_eval, que avalia uma string como uma expressão Python e retorna o objeto correspondente.

users["preferences"] = users["preferences"].apply(ast.literal_eval)
ads['categories'] = ads['categories'].apply(ast.literal_eval)

# Features Categorias

Gerar as categorias com seus pesos

OBS: isso foi feito porque para trabalhar com a abordagem de vetores, ADS e Users devem possuir mesma dimensões

Depois desenvolver uma forma mais inteligente para não fazer à mão

In [402]:
all_categories  = [
"FPS",
"Battle_Royale",
"MOBA",
"RPG",
"MMORPG",
"Strategy",
"Simulation",
"Sports_Games",
"Racing",
"Indie_Games",
"Retro_Gaming",
"Mobile_Gaming",
"PC_Gaming",
"Console_Gaming",
"Game_Development",
"Game_Streaming",
"Esports",
"Game_Hardware",
"VR_Gaming",
"Open_World",
"League_of_Legends",
"Fortnite",
"Minecraft",
"Counter_Strike",
"World_of_Warcraft"
]

In [403]:
# enumerar as categorias para criar um índice de categoria, que será usado para construir os vetores de características dos usuários e dos anúncios.
category_index = {cat: i for i, cat in enumerate(all_categories)}


In [404]:
category_index

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [405]:
# função para construir o vetor de características do usuário a partir de suas preferências, usando o índice de categoria para mapear as categorias para posições no vetor.

def build_user_vector(preferences, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in preferences:
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [406]:
# fazendo um teste para ver se a função build_user_vector está funcionando corretamente, construindo o vetor de características do primeiro usuário do dataset.
teste = build_user_vector(users.iloc[0]['preferences'], category_index)
display(teste)
display(users.iloc[0]['preferences'])
display(category_index)

array([0.        , 0.        , 0.        , 0.        , 0.17928986,
       0.        , 0.        , 0.06144034, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.16123711, 0.        ,
       0.        , 0.04964747, 0.0894841 , 0.01684707, 0.        ,
       0.        , 0.32530896, 0.03901933, 0.07772577, 0.        ])

[('Sports_Games', 0.06144033651140886),
 ('Console_Gaming', 0.16123710711415512),
 ('MMORPG', 0.17928985804561),
 ('VR_Gaming', 0.016847073302291835),
 ('Minecraft', 0.03901932791588827),
 ('Fortnite', 0.3253089551775492),
 ('Esports', 0.04964746680902405),
 ('Counter_Strike', 0.07772577069622656),
 ('Game_Hardware', 0.0894841044278461)]

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [407]:
len(teste)

25

In [408]:
# função análoga para o ads
# feito isso pq o ads foi salvo como dict e user como list

def build_ad_vector(categories, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in categories.items():
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [409]:
# teste para ads
teste_ad = build_ad_vector(ads.iloc[0]['categories'], category_index)
display(teste_ad)
display(ads.iloc[0]['categories'])


array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.09308321, 0.        , 0.        , 0.16518794,
       0.31308539, 0.        , 0.01383063, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.22099246, 0.06033233,
       0.13348803, 0.        , 0.        , 0.        , 0.        ])

{'Indie_Games': 0.16518794390557134,
 'Open_World': 0.060332325664619484,
 'PC_Gaming': 0.013830634132233021,
 'Simulation': 0.09308320786155601,
 'League_of_Legends': 0.13348803361729053,
 'VR_Gaming': 0.22099246159470362,
 'Retro_Gaming': 0.3130853932240261}

In [410]:
# aplcar as funções para construir os vetores de características para todos os usuários e anúncios, e armazenar esses vetores em novas colunas nos dataframes.

users['feature_vector'] = users['preferences'].apply(lambda prefs: build_user_vector(prefs, category_index))
ads['feature_vector'] = ads['categories'].apply(lambda cats: build_ad_vector(cats, category_index))

In [411]:
users.head(1)

,user_id,age,gender,state,city,preferences,feature_vector
0,0,58,F,MG,Juiz de Fora,"[(Sports_Games, 0.06144033651140886), (Console_Gaming, 0.16123710711415512), (MMORPG, 0.17928985804561), (VR_Gaming, 0.016847073302291835), (Minecraft, 0.03901932791588827), (Fortnite, 0.3253089551775492), (Esports, 0.04964746680902405), (Counter_Strike, 0.07772577069622656), (Game_Hardware, 0.0894841044278461)]","[0.0, 0.0, 0.0, 0.0, 0.17928985804561, 0.0, 0.0, 0.06144033651140886, 0.0, 0.0, 0.0, 0.0, 0.0, 0.16123710711415512, 0.0, 0.0, 0.04964746680902405, 0.0894841044278461, 0.016847073302291835, 0.0, 0.0, 0.3253089551775492, 0.03901932791588827, 0.07772577069622656, 0.0]"


# Features demográficas

Uma estratégia é adicionar essas features ao vetor, só que aumenta bastante a dimensionalidade.

Inicialmente, vamos fazer one-hot encoding, mas é melhor pensar em outras estratégias
-   nessas situações, há 25 cidades distintas

In [412]:
ALL_CITIES = sorted(
    set(users["city"].unique()) |
    set(ads["city_target"].unique())
)

In [413]:
ALL_STATES = sorted(
    set(users["state"].unique()) | 
    set(ads["state_target"].unique())
)

In [414]:
def one_hot_city(city, all_cities):

    vector = {f"city_{c}": 0 for c in all_cities}

    if city in all_cities:
        vector[f"city_{city}"] = 1

    return vector

In [415]:
def one_hot_state(state, all_states):

    vector = {f"state_{s}": 0 for s in all_states}

    if state in all_states:
        vector[f"state_{state}"] = 1

    return vector

In [416]:
city_features = users["city"].apply(
    lambda x: pd.Series(one_hot_city(x, ALL_CITIES))
)

city_features_ads = ads["city_target"].apply(
    lambda x: pd.Series(one_hot_city(x, ALL_CITIES))
)

In [417]:
state_features = users["state"].apply(
    lambda x: pd.Series(one_hot_state(x, ALL_STATES))
)
state_features_ads = ads["state_target"].apply(
    lambda x: pd.Series(one_hot_state(x, ALL_STATES))
)

In [418]:
users = pd.concat([users, city_features], axis=1)
ads = pd.concat([ads, city_features_ads], axis=1)

In [419]:
users = pd.concat([users, state_features], axis=1)
ads = pd.concat([ads, state_features_ads], axis=1)

### Feature genero

In [420]:
# One-hot encoding para Genero
gender_dummies = pd.get_dummies(users["gender"], prefix="gender")
users = pd.concat([users, gender_dummies], axis=1)

gender_dummies_ads = pd.get_dummies(ads["gender_target"], prefix="gender")
ads = pd.concat([ads, gender_dummies_ads], axis=1)

### Feature Idade

In [421]:
# a idade targe só tem uma idade fixa, então faz sentido normalizar.
# isso vai acontecer ou o anúncio vai ter uma faixa de valores??

In [422]:
users['age_norm'] = (users['age'] - users['age'].min()) / (users['age'].max() - users['age'].min())
ads['age_norm'] = (ads['age_target'] - users['age'].min()) / (users['age'].max() - users['age'].min())  

# Gerar os vetores finais com cidade/estado

- vai fazer a similaridade tudo junto ou vai separar??

In [423]:
# definir ordem fixa das features

CATEGORY_COLUMNS = all_categories

CITY_COLUMNS = [f"city_{c}" for c in ALL_CITIES]

STATE_COLUMNS = [f"state_{s}" for s in ALL_STATES]

GENDER_COLUMNS = ["gender_F", "gender_M"]

AGE_COLUMNS = ["age_norm"]

In [424]:
# transformar features_vector em colunas separadas
category_features_users = pd.DataFrame(
    users["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

category_features_ads = pd.DataFrame(
    ads["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

users = pd.concat([users, category_features_users], axis=1)
ads = pd.concat([ads, category_features_ads], axis=1)

In [425]:
category_features_users

,FPS,Battle_Royale,MOBA,RPG,MMORPG,Strategy,Simulation,Sports_Games,Racing,Indie_Games,...,Game_Streaming,Esports,Game_Hardware,VR_Gaming,Open_World,League_of_Legends,Fortnite,Minecraft,Counter_Strike,World_of_Warcraft
0,0.000000,0.0,0.000000,0.0,0.179290,0.000000,0.000000,0.061440,0.00000,0.0,...,0.0,0.049647,0.089484,0.016847,0.000000,0.000000,0.325309,0.039019,0.077726,0.0
1,0.000000,0.0,0.536367,0.0,0.000000,0.000000,0.000000,0.463633,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.113223,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.720546,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,0.000000,0.0,0.000000,0.0,0.257160,0.000000,0.405503,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.337338,0.000000,0.000000,0.000000,0.000000,0.0
4,0.637649,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.155128,0.000000,0.097705,0.109519,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.000000,0.0,0.000000,0.0,0.722629,0.049078,0.000000,0.000000,0.10887,0.0,...,0.0,0.000000,0.000000,0.033345,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9996,0.155263,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9997,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,...,0.0,0.000000,0.665512,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9998,0.000000,0.0,0.000000,0.0,0.000000,0.607445,0.000000,0.000000,0.00000,0.0,...,0.0,0.351253,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [426]:
users.iloc[0].preferences

[('Sports_Games', 0.06144033651140886),
 ('Console_Gaming', 0.16123710711415512),
 ('MMORPG', 0.17928985804561),
 ('VR_Gaming', 0.016847073302291835),
 ('Minecraft', 0.03901932791588827),
 ('Fortnite', 0.3253089551775492),
 ('Esports', 0.04964746680902405),
 ('Counter_Strike', 0.07772577069622656),
 ('Game_Hardware', 0.0894841044278461)]

In [427]:
users[category_features_users.columns].iloc[0]

FPS                  0.000000
Battle_Royale        0.000000
MOBA                 0.000000
RPG                  0.000000
MMORPG               0.179290
Strategy             0.000000
Simulation           0.000000
Sports_Games         0.061440
Racing               0.000000
Indie_Games          0.000000
Retro_Gaming         0.000000
Mobile_Gaming        0.000000
PC_Gaming            0.000000
Console_Gaming       0.161237
Game_Development     0.000000
Game_Streaming       0.000000
Esports              0.049647
Game_Hardware        0.089484
VR_Gaming            0.016847
Open_World           0.000000
League_of_Legends    0.000000
Fortnite             0.325309
Minecraft            0.039019
Counter_Strike       0.077726
World_of_Warcraft    0.000000
Name: 0, dtype: float64

In [428]:
# garantir que todas as colunas de cidade, estado e gênero estejam presentes em ambos os dataframes, preenchendo com zeros onde necessário
for col in GENDER_COLUMNS:
    if col not in users:
        users[col] = 0
    if col not in ads:
        ads[col] = 0

In [429]:
for col in CITY_COLUMNS:
    if col not in users:
        users[col] = 0
    if col not in ads:
        ads[col] = 0

for col in STATE_COLUMNS:
    if col not in users:
        users[col] = 0
    if col not in ads:
        ads[col] = 0

In [430]:
FINAL_FEATURE_COLUMNS = (
    CATEGORY_COLUMNS +
    CITY_COLUMNS +
    STATE_COLUMNS +
    GENDER_COLUMNS +
    AGE_COLUMNS
)

In [431]:
user_matrix = users[FINAL_FEATURE_COLUMNS].values
ad_matrix = ads[FINAL_FEATURE_COLUMNS].values

# PRECISA NORMALIZAR???

In [432]:
user_matrix

array([[0.0, 0.0, 0.0, ..., True, False, 0.9523809523809523],
       [0.0, 0.0, 0.5363673412882524, ..., True, False,
        0.5476190476190477],
       [0.0, 0.0, 0.0, ..., True, False, 0.7380952380952381],
       ...,
       [0.0, 0.0, 0.0, ..., True, False, 0.19047619047619047],
       [0.0, 0.0, 0.0, ..., False, True, 0.0],
       [0.0, 0.0, 0.0, ..., False, True, 0.6428571428571429]],
      dtype=object)

In [433]:
similarity_matrix = cosine_similarity(user_matrix, ad_matrix)

In [434]:
similarity_matrix
# cada linha representa um usuário e cada coluna representa um anúncio, e os valores são as similaridades entre eles.

array([[0.47153176, 0.24981313, 0.63731514, ..., 0.08236718, 0.15515928,
        0.15824168],
       [0.38705477, 0.12998388, 0.3446966 , ..., 0.32779277, 0.34343886,
        0.33327958],
       [0.45926929, 0.16862957, 0.40580891, ..., 0.05354563, 0.14552527,
        0.0784481 ],
       ...,
       [0.30637519, 0.04652391, 0.36367994, ..., 0.01477293, 0.03313002,
        0.02164337],
       [0.        , 0.25365693, 0.        , ..., 0.35389934, 0.25537482,
        0.28459703],
       [0.15691192, 0.92024834, 0.12374789, ..., 0.31915951, 0.34496195,
        0.33553204]])

## Validar os resultados

In [435]:
user_index = 0

scores = similarity_matrix[user_index]

top_ads = similarity_matrix[user_index].argsort()[::-1][:10]

In [436]:
top_ads

array([152, 115, 151, 338, 367, 147, 110, 330,  21, 247])

In [437]:
scores[top_ads]

array([0.94379563, 0.92677986, 0.90478853, 0.87564401, 0.87396446,
       0.86797834, 0.85945246, 0.85677553, 0.84489894, 0.82566807])

In [438]:
ads_base.iloc[top_ads]

,ad_id,state_target,city_target,age_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate
152,152,MG,Juiz de Fora,60,F,"{'Counter_Strike': 0.614514113992789, 'Console_Gaming': 0.38548588600721106}",57.504583,57504.582561,0.806772,0.028923
115,115,MG,Juiz de Fora,59,F,"{'VR_Gaming': 0.5233437094183313, 'Minecraft': 0.4766562905816686}",21.308174,2130.817443,0.961975,0.012214
151,151,MG,Juiz de Fora,42,F,"{'Game_Development': 0.5796890742480767, 'World_of_Warcraft': 0.09960639784533146, 'Minecraft': 0.3207045279065917}",33.714760,16857.380127,0.991906,0.038818
338,338,MG,Juiz de Fora,27,F,"{'PC_Gaming': 0.14144155141549086, 'Sports_Games': 0.4181766075340468, 'Open_World': 0.0125444843628015, 'Racing': 0.03440560950383214, 'Indie_Games': 0.15039945835741847, 'Mobile_Gaming': 0.2430322888264102}",20.981378,20981.377913,0.886847,0.035298
367,367,MG,Juiz de Fora,21,F,"{'PC_Gaming': 0.11208083563539241, 'Fortnite': 0.16159003121114351, 'Strategy': 0.10610049856926887, 'MMORPG': 0.44842814421598565, 'VR_Gaming': 0.0030059071099666386, 'FPS': 0.024672886263427583, 'World_of_Warcraft': 0.1441216969948152}",16.812757,16812.757363,0.644290,0.010038
147,147,MG,Juiz de Fora,42,F,"{'PC_Gaming': 0.14857127535626957, 'Battle_Royale': 0.8514287246437305}",15.000000,1500.000000,0.578580,0.009379
110,110,MG,Juiz de Fora,43,F,{'Sports_Games': 1.0},15.000000,7500.000000,0.726242,0.035685
330,330,MG,Juiz de Fora,23,F,"{'Sports_Games': 0.4177617503847425, 'Racing': 0.012445183557036705, 'VR_Gaming': 0.02295918269455457, 'MMORPG': 0.5468338833636663}",19.775528,9887.763938,0.870138,0.030988
21,21,MG,Juiz de Fora,43,F,{'Game_Streaming': 1.0},31.879192,31879.192117,0.942092,0.027232
247,247,MG,Juiz de Fora,18,F,"{'Retro_Gaming': 0.4630104530021115, 'Indie_Games': 0.07764622039070125, 'Sports_Games': 0.08454687553195456, 'Console_Gaming': 0.3747964510752326}",21.618254,21618.254103,0.850561,0.011832


In [439]:
users.loc[0]

user_id                         0
age                            58
gender                          F
state                          MG
city                 Juiz de Fora
                         ...     
League_of_Legends             0.0
Fortnite                 0.325309
Minecraft                0.039019
Counter_Strike           0.077726
World_of_Warcraft             0.0
Name: 0, Length: 65, dtype: object

# Gerar os vetores finais sem cidade/estado

In [440]:
users_no_location = users
ads_no_location = ads

In [441]:
category_features_users

,FPS,Battle_Royale,MOBA,RPG,MMORPG,Strategy,Simulation,Sports_Games,Racing,Indie_Games,...,Game_Streaming,Esports,Game_Hardware,VR_Gaming,Open_World,League_of_Legends,Fortnite,Minecraft,Counter_Strike,World_of_Warcraft
0,0.000000,0.0,0.000000,0.0,0.179290,0.000000,0.000000,0.061440,0.00000,0.0,...,0.0,0.049647,0.089484,0.016847,0.000000,0.000000,0.325309,0.039019,0.077726,0.0
1,0.000000,0.0,0.536367,0.0,0.000000,0.000000,0.000000,0.463633,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.113223,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.720546,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,0.000000,0.0,0.000000,0.0,0.257160,0.000000,0.405503,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.337338,0.000000,0.000000,0.000000,0.000000,0.0
4,0.637649,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.155128,0.000000,0.097705,0.109519,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.000000,0.0,0.000000,0.0,0.722629,0.049078,0.000000,0.000000,0.10887,0.0,...,0.0,0.000000,0.000000,0.033345,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9996,0.155263,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9997,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,...,0.0,0.000000,0.665512,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
9998,0.000000,0.0,0.000000,0.0,0.000000,0.607445,0.000000,0.000000,0.00000,0.0,...,0.0,0.351253,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [442]:
users_no_location.iloc[0].preferences

[('Sports_Games', 0.06144033651140886),
 ('Console_Gaming', 0.16123710711415512),
 ('MMORPG', 0.17928985804561),
 ('VR_Gaming', 0.016847073302291835),
 ('Minecraft', 0.03901932791588827),
 ('Fortnite', 0.3253089551775492),
 ('Esports', 0.04964746680902405),
 ('Counter_Strike', 0.07772577069622656),
 ('Game_Hardware', 0.0894841044278461)]

In [443]:
users_no_location[category_features_users.columns].iloc[0]

FPS                  0.000000
Battle_Royale        0.000000
MOBA                 0.000000
RPG                  0.000000
MMORPG               0.179290
Strategy             0.000000
Simulation           0.000000
Sports_Games         0.061440
Racing               0.000000
Indie_Games          0.000000
Retro_Gaming         0.000000
Mobile_Gaming        0.000000
PC_Gaming            0.000000
Console_Gaming       0.161237
Game_Development     0.000000
Game_Streaming       0.000000
Esports              0.049647
Game_Hardware        0.089484
VR_Gaming            0.016847
Open_World           0.000000
League_of_Legends    0.000000
Fortnite             0.325309
Minecraft            0.039019
Counter_Strike       0.077726
World_of_Warcraft    0.000000
Name: 0, dtype: float64

In [444]:
FINAL_FEATURE_COLUMNS = (
    CATEGORY_COLUMNS +
    AGE_COLUMNS
)

In [445]:
user_matrix = users_no_location[FINAL_FEATURE_COLUMNS].values
ad_matrix = ads_no_location[FINAL_FEATURE_COLUMNS].values

# PRECISA NORMALIZAR???

In [446]:
user_matrix

array([[0.        , 0.        , 0.        , ..., 0.07772577, 0.        ,
        0.95238095],
       [0.        , 0.        , 0.53636734, ..., 0.        , 0.        ,
        0.54761905],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.73809524],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.19047619],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.69859841, 0.        ,
        0.64285714]])

In [447]:
similarity_matrix = cosine_similarity(user_matrix, ad_matrix)

In [448]:
similarity_matrix
# cada linha representa um usuário e cada coluna representa um anúncio, e os valores são as similaridades entre eles.

array([[0.83325925, 0.84722883, 0.58315948, ..., 0.35103376, 0.53378324,
        0.7880392 ],
       [0.55699446, 0.49588724, 0.39133844, ..., 0.30680056, 0.38172655,
        0.33874517],
       [0.78715307, 0.5697446 , 0.57113993, ..., 0.22734206, 0.49875423,
        0.38919781],
       ...,
       [0.22574851, 0.20098191, 0.49284051, ..., 0.0801967 , 0.14517911,
        0.1372926 ],
       [0.        , 0.        , 0.        , ..., 0.4609661 , 0.        ,
        0.        ],
       [0.58807091, 0.73240811, 0.41317243, ..., 0.27259929, 0.37818904,
        0.35764482]])

## Validar os resultados

In [449]:
user_index = 0

user_city = users_no_location.iloc[user_index]["city"]
user_gender = users_no_location.iloc[user_index]["gender"]

scores = similarity_matrix[user_index]

ads_filtered = ads_no_location[
    (ads_no_location["city_target"] == user_city) &
    (ads_no_location["gender_target"] == user_gender)
]

filtered_scores = scores[ads_filtered.index]

top_ads = ads_filtered.index[filtered_scores.argsort()[::-1][:10]]

In [450]:
top_ads

Index([152, 115, 151, 110, 147, 21, 338, 367, 330, 247], dtype='int64')

In [451]:
scores[top_ads]

array([0.82244872, 0.75919001, 0.60474229, 0.51642013, 0.50238437,
       0.46592304, 0.38620739, 0.36502444, 0.32501104, 0.10346772])

In [452]:
ads_base.iloc[top_ads]

,ad_id,state_target,city_target,age_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate
152,152,MG,Juiz de Fora,60,F,"{'Counter_Strike': 0.614514113992789, 'Console_Gaming': 0.38548588600721106}",57.504583,57504.582561,0.806772,0.028923
115,115,MG,Juiz de Fora,59,F,"{'VR_Gaming': 0.5233437094183313, 'Minecraft': 0.4766562905816686}",21.308174,2130.817443,0.961975,0.012214
151,151,MG,Juiz de Fora,42,F,"{'Game_Development': 0.5796890742480767, 'World_of_Warcraft': 0.09960639784533146, 'Minecraft': 0.3207045279065917}",33.714760,16857.380127,0.991906,0.038818
110,110,MG,Juiz de Fora,43,F,{'Sports_Games': 1.0},15.000000,7500.000000,0.726242,0.035685
147,147,MG,Juiz de Fora,42,F,"{'PC_Gaming': 0.14857127535626957, 'Battle_Royale': 0.8514287246437305}",15.000000,1500.000000,0.578580,0.009379
21,21,MG,Juiz de Fora,43,F,{'Game_Streaming': 1.0},31.879192,31879.192117,0.942092,0.027232
338,338,MG,Juiz de Fora,27,F,"{'PC_Gaming': 0.14144155141549086, 'Sports_Games': 0.4181766075340468, 'Open_World': 0.0125444843628015, 'Racing': 0.03440560950383214, 'Indie_Games': 0.15039945835741847, 'Mobile_Gaming': 0.2430322888264102}",20.981378,20981.377913,0.886847,0.035298
367,367,MG,Juiz de Fora,21,F,"{'PC_Gaming': 0.11208083563539241, 'Fortnite': 0.16159003121114351, 'Strategy': 0.10610049856926887, 'MMORPG': 0.44842814421598565, 'VR_Gaming': 0.0030059071099666386, 'FPS': 0.024672886263427583, 'World_of_Warcraft': 0.1441216969948152}",16.812757,16812.757363,0.644290,0.010038
330,330,MG,Juiz de Fora,23,F,"{'Sports_Games': 0.4177617503847425, 'Racing': 0.012445183557036705, 'VR_Gaming': 0.02295918269455457, 'MMORPG': 0.5468338833636663}",19.775528,9887.763938,0.870138,0.030988
247,247,MG,Juiz de Fora,18,F,"{'Retro_Gaming': 0.4630104530021115, 'Indie_Games': 0.07764622039070125, 'Sports_Games': 0.08454687553195456, 'Console_Gaming': 0.3747964510752326}",21.618254,21618.254103,0.850561,0.011832


In [453]:
users.loc[0]

user_id                         0
age                            58
gender                          F
state                          MG
city                 Juiz de Fora
                         ...     
League_of_Legends             0.0
Fortnite                 0.325309
Minecraft                0.039019
Counter_Strike           0.077726
World_of_Warcraft             0.0
Name: 0, Length: 65, dtype: object

# **OBJETIVO : score final**

score =
interest_similarity
× demographic_score
× bid
× trust_score
× estimated_conversion_rate

NÃO ESQUECER: incluir as